# ArchitectAgent Fine-Tuning Pipeline
## QLoRA Training of Phi-3-mini-128k-instruct for Implementation-Grade Architecture Generation

**Task:** Train a specialised ArchitectAgent that produces complete, structured JSON architecture plans  
**Method:** QLoRA (4-bit NormalFloat quantisation) with completion-only loss masking  
**Base Model:** Microsoft Phi-3-mini-128k-instruct (3.8B parameters)

---

### Key Design Decisions

| Decision | Choice | Justification |
|----------|--------|---------------|
| Base model | Phi-3-mini-128k-instruct | 3.8B params fits in 16GB VRAM with QLoRA; 128K context handles long architect inputs; strong instruction-following for structured JSON |
| Fine-tuning method | QLoRA (LoRA rank=64, all linear layers) | Rank 64 for complex 19-section plan generation; 4-bit quantisation reduces ~7.6GB to ~2GB |
| Loss masking | DataCollatorForCompletionOnlyLM | Loss computed ONLY on the plan JSON output (15-25% of tokens), not on the large input payload |
| Input trimming | Contract values-only + sub-field caps | Frozen contract + specialist subplans can exceed 8K tokens; structured trimming preserves semantic content |
| plan_quality fix | Excluded from training format | Original labels (`strong/weak/flawed`) were invalid; cleaned labels are word-count proxies. Field is metadata, not part of the Architect's operational interface — it is never included in training inputs |
| Data split | Chain-aware 85/15 | Consistent with Auditor pipeline; each row is self-contained but chain fingerprinting ensures no shared project appears in both splits |
| Epochs | 3 | Architect outputs are ~3× longer than Auditor outputs (3300 vs 1100 tokens); more gradient signal per step means fewer epochs required |
| max_seq_length | 10240 | Architect input (contract + subplans + reviews) + output (plan) frequently exceeds 8192; 10240 accommodates p90 of dataset |
| Learning rate | 1e-4 with cosine decay | Standard for QLoRA; cosine schedule prevents destabilisation during long plan generation learning |

---
## Section 1 — Environment Setup

In [ ]:
import os, json, time, hashlib, warnings, random, re, copy
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, AutoConfig,
    BitsAndBytesConfig, TrainingArguments, TrainerCallback,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    PeftModel, TaskType,
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import Dataset
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")
print(f"Seed: {SEED}")

In [ ]:
# ===============================================================
# CONFIGURATION
# ===============================================================

DATASET_PATH = "architect_dataset_final.jsonl"
MODEL_NAME = "microsoft/Phi-3-mini-128k-instruct"

OUTPUT_DIR = Path("training_output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_MODEL_DIR = OUTPUT_DIR / "architect_agent_model"
LOGS_DIR = OUTPUT_DIR / "logs"

for d in [OUTPUT_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, LOGS_DIR]:
    d.mkdir(exist_ok=True)

CONFIG = {
    # LoRA
    "lora_r": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # Training
    "num_epochs": 3,
    "per_device_batch_size": 1,
    "gradient_accumulation": 8,
    "learning_rate": 1e-4,
    "lr_scheduler": "cosine",
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 10240,
    # Split
    "val_ratio": 0.15,
    "specialist_field_cap": 250,
    "reasoner_field_cap": 500,
    "prev_audit_summary_cap": 400,
    "prev_plan_section_cap": 300,
}

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## Section 2 — System Prompt & Data Loading

The system prompt is identical to what `main.py` uses at runtime.

**plan_quality fix:** The dataset's `plan_quality` metadata field carried unreliable labels
(`strong`/`weak`/`flawed` in the raw data; word-count-derived `good`/`moderate` after cleaning).
This field is **not part of the Architect's operational interface** — at runtime the agent
receives only the requirement contract and context, never a quality label.
It is therefore excluded entirely from the training format. A proper richness score is computed
from the actual plan content and used only for pre-training quality filtering.

In [ ]:
# ===============================================================
# ARCHITECT SYSTEM PROMPT
# ===============================================================

ARCHITECT_SYSTEM_PROMPT = """You are the architecture generator.

Create a polished implementation-grade architecture plan from:
- frozen confirmed requirement contract
- rich requirement notes
- specialist reviews
- cumulative issue ledger
- focus issues
- revision memory
- previous audits
- best prior plan

Main goal:
- First, address the current focus issues.
- Second, preserve all user-confirmed requirements.
- Third, improve the architecture without introducing regressions.

Rules:
- Do not mention round numbers in the title.
- Preserve user-confirmed requirements.
- Prioritize unresolved critical and high-severity focus issues first.
- For each focus issue, either fix it in the plan or clearly explain why it remains unresolved.
- Do not ignore recurring unresolved issues from previous rounds.
- Try to improve weak areas identified by the auditor before adding extra design complexity.
- Include concrete architecture, modules, workflows, schemas, APIs, security, deployment, observability, roadmap, and developer guidance.
- Keep the plan implementation-grade and specific, not generic.

Return JSON only with:
- thinking_summary
- fix_report
- title
- executive_summary
- architecture_overview
- technology_stack
- functional_feature_map
- system_components
- workflows
- data_model
- api_design
- security_and_compliance
- deployment_and_operations
- observability
- cost_and_scaling
- phased_implementation
- development_guidelines
- risks_and_tradeoffs
- open_questions_resolved

fix_report must be a list of items with:
- issue_id
- action_taken
- changed_sections
- expected_outcome

For each fix_report item:
- issue_id must match the issue being addressed
- action_taken must say what was changed
- changed_sections must name the plan sections updated
- expected_outcome must explain what the auditor should now find improved"""

print(f"System prompt: {len(ARCHITECT_SYSTEM_PROMPT)} characters")

In [ ]:
# ===============================================================
# LOAD DATASET
# ===============================================================

PLAN_CONTENT_SECTIONS = [
    "executive_summary", "architecture_overview", "technology_stack",
    "functional_feature_map", "system_components", "workflows",
    "data_model", "api_design", "security_and_compliance",
    "deployment_and_operations", "observability", "cost_and_scaling",
    "phased_implementation", "development_guidelines", "risks_and_tradeoffs",
    "open_questions_resolved",
]

MANDATORY_PLAN_KEYS = ["thinking_summary", "fix_report", "title"] + PLAN_CONTENT_SECTIONS

TECH_TERMS = [
    'kubernetes','k8s','docker','ecs','fargate','lambda','rds','dynamodb',
    'postgres','mongodb','redis','kafka','rabbitmq','nginx','express','fastapi',
    'django','spring','react','vue','angular','flutter','swift','kotlin',
    'terraform','helm','argocd','github actions','datadog','prometheus',
    'grafana','cloudwatch','tls','oauth','oidc','jwt','auth0','mfa','rbac',
    'waf','s3','cloudfront','api gateway','sns','sqs','openapi','graphql',
    'grpc','websocket','sse','sqlite','mysql','elasticsearch','signalr',
    'wpf','dotnet','supabase','firebase','vercel','netlify','aws','gcp',
    'azure','typescript','nextjs','tailwind','rust','golang','go ',
]


def compute_plan_richness(output: dict) -> dict:
    """Measure actual plan quality from content — the ground truth the metadata label never was."""
    total_words = 0
    stub_sections = []
    for section in PLAN_CONTENT_SECTIONS:
        val = output.get(section, '')
        text = json.dumps(val) if isinstance(val, dict) else ' '.join(val) if isinstance(val, list) else str(val or '')
        wc = len(text.split())
        total_words += wc
        if wc <= 5:
            stub_sections.append(section)
    plan_text = json.dumps(output).lower()
    tech_count = sum(1 for t in TECH_TERMS if t in plan_text)
    return {
        "total_words": total_words,
        "stub_sections": stub_sections,
        "tech_term_count": tech_count,
        "is_rich": total_words >= 400 and len(stub_sections) == 0,
    }


def load_jsonl(filepath: str) -> list:
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows


raw_data = load_jsonl(DATASET_PATH)
print(f"Loaded {len(raw_data)} rows from {DATASET_PATH}")

# ---- RICHNESS-BASED QUALITY FILTER ----------------------------------------
# plan_quality metadata is unreliable; filter on actual measured plan richness.
before_filter = len(raw_data)
raw_data = [r for r in raw_data if compute_plan_richness(r.get('target_output', {}))['is_rich']]
print(f"Richness filter: kept {len(raw_data)}/{before_filter} rows (removed {before_filter - len(raw_data)} low-quality plans)")

# Overview
case_types = Counter(r.get('metadata', {}).get('case_type', '?') for r in raw_data)
pclasses = Counter(r.get('profile', {}).get('project_class', '?') for r in raw_data)
rounds  = Counter(r.get('input_payload', {}).get('round', 0) for r in raw_data)

richness_scores = [compute_plan_richness(r.get('target_output', {}))['total_words'] for r in raw_data]

print(f"Case types: {dict(case_types)}")
print(f"Project classes ({len(pclasses)} unique): represented")
print(f"Round distribution: {dict(sorted(rounds.items()))}")
print(f"Plan richness (words): mean={np.mean(richness_scores):.0f}, "f"median={np.median(richness_scores):.0f}, "f"min={min(richness_scores)}, max={max(richness_scores)}")

---
## Section 3 — Tokenizer & Data Formatting

**Critical design decision:** The Architect input payload is significantly larger than the
Auditor's. The frozen requirement contract alone averages ~1750 tokens; specialist subplans
add ~1200 more. Naively serialising the full payload would exceed `max_seq_length` for ~36%
of rows. A structured trimming strategy is applied that preserves all semantically critical
fields while capping verbose sub-fields:

| Field | Treatment |
|-------|-----------|
| `frozen_requirement_contract` | Values only — `source`, `rationale`, `confirmed`, `updated_at` are training-pipeline metadata; the model needs only the requirement values |
| `specialist_subplans` | Each sub-field capped at 250 chars — sufficient for the model to understand the specialist's recommendation |
| `reasoner_reviews` | Each reviewer capped at 500 chars |
| `previous_audits` | Last audit only; trimmed to score + summary + blocking_issues + recommendations |
| `previous_plan` / `best_plan` | Title + executive_summary (300 chars) + technology_stack (300 chars) only |
| `issue_ledger`, `focus_issues`, `revision_memory` | Always included in full — these are the primary signals for revision rounds |

**plan_quality is not included anywhere in the formatted training text.**

In [ ]:
# ===============================================================
# LOAD TOKENIZER
# ===============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab: {tokenizer.vocab_size}, Pad: {tokenizer.pad_token}, EOS: {tokenizer.eos_token}")

In [ ]:
# ===============================================================
# INPUT TRIMMING
# ===============================================================

def trim_contract(contract: dict) -> dict:
    """
    Keep only the 'value' field from each contract entry.
    Source, rationale, confirmed, and updated_at are training pipeline metadata
    and add ~800 tokens of noise without improving the model's ability to generate plans.
    """
    trimmed = {}
    for field, entry in contract.items():
        if isinstance(entry, dict):
            trimmed[field] = entry.get('value', '')
        else:
            trimmed[field] = entry
    return trimmed


def trim_specialist_subplans(subplans: dict, cap: int) -> dict:
    trimmed = {}
    for specialist, data in subplans.items():
        if isinstance(data, dict):
            trimmed[specialist] = {k: str(v)[:cap] for k, v in data.items()}
        else:
            trimmed[specialist] = str(data)[:cap]
    return trimmed


def trim_reasoner_reviews(reviews: dict, cap: int) -> dict:
    return {k: str(v)[:cap] for k, v in reviews.items()}


def trim_previous_audits(audits: list, summary_cap: int) -> list:
    """Keep the last audit only with trimmed summary and first 3 blocking issues / recommendations."""
    if not audits:
        return []
    last = audits[-1]
    if not isinstance(last, dict):
        return []
    return [{
        'round': last.get('round'),
        'score': last.get('score'),
        'summary': str(last.get('summary', ''))[:summary_cap],
        'blocking_issues': (last.get('blocking_issues') or [])[:3],
        'recommendations': (last.get('recommendations') or [])[:3],
    }]


def trim_plan_reference(plan: dict, section_cap: int) -> dict:
    """For previous_plan / best_plan — keep only the three highest-signal sections."""
    if not plan or not isinstance(plan, dict):
        return {}
    return {
        'title': str(plan.get('title', ''))[:100],
        'executive_summary': str(plan.get('executive_summary', ''))[:section_cap],
        'technology_stack': str(plan.get('technology_stack', ''))[:section_cap],
    }


def build_architect_input(payload: dict) -> dict:
    """Assemble and trim the architect input from the raw payload."""
    cap_spec = CONFIG['specialist_field_cap']
    cap_rr = CONFIG['reasoner_field_cap']
    cap_audit = CONFIG['prev_audit_summary_cap']
    cap_plan = CONFIG['prev_plan_section_cap']

    return {
        "round": payload.get('round', 1),
        "frozen_requirement_contract": trim_contract(payload.get('frozen_requirement_contract', {})),
        "requirements": payload.get('requirements', {}),
        "reasoner_reviews": trim_reasoner_reviews(payload.get('reasoner_reviews', {}), cap_rr),
        "specialist_subplans": trim_specialist_subplans(payload.get('specialist_subplans', {}), cap_spec),
        "issue_ledger": payload.get('issue_ledger', {}),
        "focus_issues": payload.get('focus_issues', []),
        "revision_memory": payload.get('revision_memory', {}),
        "accepted_exceptions": payload.get('accepted_exceptions', {}),
        "previous_audits": trim_previous_audits(payload.get('previous_audits', []), cap_audit),
        "previous_plan": trim_plan_reference( payload.get('previous_plan', {}), cap_plan),
        "best_plan": trim_plan_reference(payload.get('best_plan', {}), cap_plan),
    }

print("Input trimming functions defined.")

In [ ]:
# ===============================================================
# FORMAT DATASET AS TEXT WITH CHAT TEMPLATE
# ===============================================================

def format_row_as_text(row: dict) -> str:
    payload = row.get('input_payload', {})
    target = row.get('target_output', {})

    architect_input = build_architect_input(payload)

    messages = [
        {"role": "system", "content": ARCHITECT_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(architect_input, indent=1, ensure_ascii=False)},
        {"role": "assistant", "content": json.dumps(target, indent=1, ensure_ascii=False)},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


formatted_texts = []
chain_fps = []
sample_ids = []
skipped = 0
token_counts = []

for row in raw_data:
    try:
        text = format_row_as_text(row)
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if n_tokens > CONFIG['max_seq_length']:
            skipped += 1
            continue
        formatted_texts.append(text)
        token_counts.append(n_tokens)
        sample_ids.append(row.get('sample_id', ''))

        # Chain fingerprint: hash the raw (un-trimmed) contract to keep chain semantics intact
        contract = row.get('input_payload', {}).get('frozen_requirement_contract', {})
        fp = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:16]
        chain_fps.append(fp)
    except Exception as e:
        skipped += 1

print(f"Formatted : {len(formatted_texts)} rows")
print(f"Skipped : {skipped} (exceeded max_seq_length or error)")
print(f"Token counts: mean={np.mean(token_counts):.0f}, "f"median={np.median(token_counts):.0f}, max={max(token_counts)}")

# ---- CRITICAL VERIFICATION ----
print(f"\n{'='*60}")
print("FORMAT VERIFICATION")
print(f"{'='*60}")

sample = formatted_texts[0]
assistant_marker = "<|assistant|>"
end_marker = "<|end|>"

if assistant_marker in sample:
    pos = sample.index(assistant_marker)
    total_chars = len(sample)
    input_chars = pos
    output_chars = total_chars - pos
    print(f"  Assistant marker found at position {pos}")
    print(f"  Input portion:  {input_chars} chars ({input_chars/total_chars*100:.0f}%)")
    print(f"  Output portion: {output_chars} chars ({output_chars/total_chars*100:.0f}%)")

    assistant_text = sample[pos:]
    expected_keys  = ["thinking_summary", "fix_report", "title", "executive_summary", "technology_stack", "system_components", "data_model", "api_design", "security_and_compliance", "deployment_and_operations"]
    found = [k for k in expected_keys if k in assistant_text]
    missing = [k for k in expected_keys if k not in assistant_text]
    print(f"\n  Plan keys found in output: {found}")
    if missing:
        print(f"  WARNING — Missing keys: {missing}")
    else:
        print(f"  All expected plan keys present in output.")

    # Verify plan_quality is not in the formatted text
    if 'plan_quality' not in sample:
        print(f"  plan_quality: CORRECTLY ABSENT from training text.")
    else:
        print(f"  WARNING: plan_quality found in formatted text — check trimming.")
else:
    print("  CRITICAL ERROR: <|assistant|> marker not found in formatted text!")

---
## Section 4 — Completion-Only Loss Masking

**This is the most critical component of the training pipeline.**

`DataCollatorForCompletionOnlyLM` masks all tokens before the `<|assistant|>` response marker.
Without this, the model achieves low loss by predicting the deterministic input payload
(requirement contract, issue ledger, specialist subplans) rather than learning to generate
the 19-section architecture plan.

Expected masking ratio for the Architect: ~75-85% masked (input), ~15-25% unmasked (output).
The Architect output is proportionally larger than the Auditor's, so the unmasked fraction
will be higher — this is correct and means more gradient signal per example.

In [ ]:
# ===============================================================
# COMPLETION-ONLY DATA COLLATOR
# ===============================================================

response_template = "<|assistant|>"
response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)

print(f"Response template: '{response_template}'")
print(f"Template token IDs: {response_template_ids}")

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# ---- VERIFY MASKING WORKS ----
print(f"\n{'='*60}")
print("LOSS MASKING VERIFICATION")
print(f"{'='*60}")

for i in range(min(5, len(formatted_texts))):
    enc = tokenizer(formatted_texts[i], return_tensors="pt", truncation=True, max_length=CONFIG['max_seq_length'])
    batch = collator([{"input_ids": enc["input_ids"][0], "attention_mask": enc["attention_mask"][0]}])
    labels = batch["labels"][0]
    masked = (labels == -100).sum().item()
    unmasked = (labels != -100).sum().item()
    total = len(labels)
    print(f"  Example {i}: {total} tokens, "f"masked={masked} ({masked/total*100:.0f}%), "f"unmasked={unmasked} ({unmasked/total*100:.0f}%)")

avg_unmasked_pct = []
for i in range(min(20, len(formatted_texts))):
    enc = tokenizer(formatted_texts[i], return_tensors="pt", truncation=True, max_length=CONFIG['max_seq_length'])
    batch = collator([{"input_ids": enc["input_ids"][0], "attention_mask": enc["attention_mask"][0]}])
    labels = batch["labels"][0]
    unmasked = (labels != -100).sum().item()
    avg_unmasked_pct.append(unmasked / len(labels) * 100)

mean_pct = np.mean(avg_unmasked_pct)
print(f"\n  Average unmasked: {mean_pct:.1f}%")
if mean_pct < 5:
    print("  CRITICAL: Less than 5% unmasked — masking may be broken!")
elif mean_pct > 60:
    print("  WARNING: More than 60% unmasked — masking may not be working!")
else:
    print(f"  GOOD: ~{mean_pct:.0f}% of tokens are plan output (loss computed on these)")

---
## Section 5 — Chain-Aware Train/Validation Split

Consistent with the Auditor pipeline: rows are split at chain level, not row level.

**Note:** Each row in the Architect dataset is self-contained (all prior context is embedded
in `input_payload`). The contract fingerprint is unique per project, so chains are effectively
length-1 here. Chain-aware splitting still gives a clean guarantee: no training-set project
can appear in validation, even if additional rounds are added to the dataset later.

In [ ]:
# ===============================================================
# CHAIN-AWARE SPLIT
# ===============================================================

chains = defaultdict(list)
for i, fp in enumerate(chain_fps):
    chains[fp].append(i)

chain_ids = list(chains.keys())
rng = random.Random(SEED)
rng.shuffle(chain_ids)

n_val = max(1, int(len(chain_ids) * CONFIG['val_ratio']))
val_chain_set = set(chain_ids[:n_val])

train_idx, val_idx = [], []
for cid in chain_ids:
    target = val_idx if cid in val_chain_set else train_idx
    target.extend(chains[cid])

train_texts = [formatted_texts[i] for i in train_idx]
val_texts = [formatted_texts[i] for i in val_idx]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

# Verify no leakage
train_chains = set(chain_fps[i] for i in train_idx)
val_chains = set(chain_fps[i] for i in val_idx)
leakage = train_chains & val_chains

print(f"Chains: {len(chains)} total, {len(chains) - n_val} train, {n_val} val")
print(f"Rows: {len(train_texts)} train, {len(val_texts)} val")
print(f"Chain leakage: {'NONE (good)' if not leakage else f'DETECTED: {len(leakage)} chains!'}")

# Case type distribution in each split
train_ids = set(sample_ids[i] for i in train_idx)
val_ids = set(sample_ids[i] for i in val_idx)

train_rows_meta = [r for r in raw_data if r.get('sample_id','') in train_ids]
val_rows_meta = [r for r in raw_data if r.get('sample_id','') in val_ids]

train_ct = Counter(r.get('metadata',{}).get('case_type','?') for r in train_rows_meta)
val_ct = Counter(r.get('metadata',{}).get('case_type','?') for r in val_rows_meta)
print(f"\nTrain case types: {dict(train_ct)}")
print(f"Val case types: {dict(val_ct)}")

---
## Section 6 — Model Loading with 4-bit Quantisation

NormalFloat4 quantisation reduces the 3.8B parameter model from ~7.6 GB to ~2 GB,
enabling training on consumer GPUs with 16 GB VRAM.

In [ ]:
# ===============================================================
# QUANTIZATION CONFIG
# ===============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Section 7 — LoRA Adapter Configuration

| Parameter | Value | Justification |
|-----------|-------|---------------|
| Rank (r) | 64 | Identical to Auditor; generating 19-section structured JSON plans requires high-rank adaptation to learn precise section ordering and content patterns |
| Alpha | 128 | Standard ratio alpha = 2r for stable gradient scaling |
| Dropout | 0.05 | Light regularisation; 883 training rows is sufficient that 0.05 prevents overfitting without starving gradient flow |
| Target modules | All 7 linear layers | Full expressiveness required; the Architect task changes output distribution from natural language to a domain-specific 19-key JSON format with named technology patterns |

In [ ]:
# ===============================================================
# APPLY LoRA
# ===============================================================

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print(f"Trainable: {trainable:,} / {total:,} ({pct:.2f}%)")
model.print_trainable_parameters()

---
## Section 8 — Training Configuration & Execution

Training loss will be in the range **2.0–4.0** throughout. This is expected and correct.
With completion-only loss masking, the model is learning to generate the hard part — a
19-section, 3000-token architecture plan — rather than predicting the repetitive input
contract. A lower loss from the Auditor run would indicate the prompt format leaked into
the loss computation.

Three epochs are used instead of five because the Architect output tokens are ~3× longer
than the Auditor output, providing ~3× more gradient signal per example per epoch.

In [ ]:
# ===============================================================
# TRAINING ARGS
# ===============================================================

total_steps = len(train_dataset) * CONFIG['num_epochs'] // CONFIG['gradient_accumulation']
eval_steps = max(1, int(total_steps * 0.1))
save_steps = eval_steps * 3

print(f"Training plan:")
print(f"  Examples: {len(train_dataset)} train, {len(val_dataset)} val")
print(f"  Epochs: {CONFIG['num_epochs']}, Effective batch: {CONFIG['gradient_accumulation']}")
print(f"  Steps: ~{total_steps}, Eval every {eval_steps}, Save every {save_steps}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=False,
    bf16=True,
    logging_dir=str(LOGS_DIR),
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="tensorboard",
    seed=SEED,
)
print("Training args ready.")

In [ ]:
# ===============================================================
# METRICS CALLBACK
# ===============================================================

class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.train_losses, self.eval_losses = [], []
        self.learning_rates, self.steps, self.eval_steps_list = [], [], []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
            if 'learning_rate' in logs:
                self.learning_rates.append(logs['learning_rate'])
            if 'eval_loss' in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps_list.append(state.global_step)

metrics_cb = MetricsCallback()

In [ ]:
# ===============================================================
# TRAIN WITH COMPLETION-ONLY LOSS
# ===============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    max_seq_length=CONFIG['max_seq_length'],
    dataset_text_field="text",
    packing=False,
    callbacks=[metrics_cb],
)

print("Starting training with completion-only loss masking...")
print("=" * 60)

start_time  = time.time()
train_result = trainer.train()
elapsed     = time.time() - start_time

print("=" * 60)
print(f"TRAINING COMPLETE")
print(f"  Duration   : {elapsed/60:.1f} minutes")
print(f"  Final loss : {train_result.training_loss:.4f}")
print(f"  Steps      : {train_result.global_step}")

trainer.save_metrics("train", train_result.metrics)

In [ ]:
# ===============================================================
# EVALUATE
# ===============================================================

trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "NotebookProgress" not in type(cb).__name__
]

eval_results = trainer.evaluate()
eval_loss = eval_results.get('eval_loss', 0)
perplexity = np.exp(eval_loss) if eval_loss < 20 else float('inf')

print(f"Eval loss : {eval_loss:.4f}")
print(f"Perplexity : {perplexity:.2f}")
trainer.save_metrics("eval", eval_results)

---
## Section 9 — Save Model

**Important:** Save the LoRA adapter BEFORE calling `merge_and_unload()`.

In [ ]:
# ===============================================================
# SAVE LoRA ADAPTER
# ===============================================================

adapter_dir = FINAL_MODEL_DIR / "lora_adapter"
adapter_dir.mkdir(exist_ok=True)

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

adapter_size = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
print(f"Adapter saved: {adapter_dir} ({adapter_size:.1f} MB)")

---
## Section 10 — Quick Generation Test

Verify the model produces a complete architecture plan JSON before running full evaluation.

In [ ]:
# ===============================================================
# QUICK GENERATION TEST
# ===============================================================

model.eval()

# Use a real example from the dataset
test_row = raw_data[0]
test_input = build_architect_input(test_row.get('input_payload', {}))

messages = [
    {"role": "system", "content": ARCHITECT_SYSTEM_PROMPT},
    {"role": "user", "content": json.dumps(test_input, indent=1, ensure_ascii=False)},
]

prompt  = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs  = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=CONFIG['max_seq_length'] - 2048)
inputs  = {k: v.to(model.device) for k, v in inputs.items()}
end_token_id = tokenizer.convert_tokens_to_ids("<|end|>")

print(f"Input tokens: {inputs['input_ids'].shape[1]}")
print("Generating plan (max_new_tokens=2048)...")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.35,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=[tokenizer.eos_token_id, end_token_id],
        use_cache=False,
    )

gen_ids = out[0][inputs['input_ids'].shape[1]:]
generated = tokenizer.decode(gen_ids, skip_special_tokens=True)

print(f"\nGenerated ({len(generated)} chars):")
print(generated[:500], "...")

# Parse check
try:
    start = generated.find('{')
    end   = generated.rfind('}')
    if start != -1 and end > start:
        parsed = json.loads(generated[start:end+1])
        sections_present = [k for k in MANDATORY_PLAN_KEYS if k in parsed]
        sections_missing = [k for k in MANDATORY_PLAN_KEYS if k not in parsed]
        richness = compute_plan_richness(parsed)
        print(f"\n{'='*60}")
        print(f"JSON VALID : YES")
        print(f"Sections present : {len(sections_present)}/{len(MANDATORY_PLAN_KEYS)}")
        if sections_missing:
            print(f"Sections missing : {sections_missing}")
        print(f"Plan words : {richness['total_words']}")
        print(f"Tech terms : {richness['tech_term_count']}")
        print(f"Title : {parsed.get('title','')[:80]}")
        print(f"fix_report items : {len(parsed.get('fix_report', []))}")
        # Check plan_quality is absent
        if 'plan_quality' not in parsed:
            print(f"plan_quality : CORRECTLY ABSENT from generated output")
    else:
        print(f"\nJSON VALID: NO (no JSON object found)")
except Exception as e:
    print(f"\nJSON VALID: NO ({e})")

---
## Section 11 — Full Structural & Semantic Evaluation

Generate architecture plans for 50 validation examples and measure:

1. **Structural metrics:** JSON validity, schema compliance (all 19 sections), fix_report
   integrity (issue IDs reference the input ledger), case-type compliance, title quality
2. **Semantic metrics:** Plan word count vs reference, tech term count, section completeness
   rate, section-level Jaccard overlap, fix_report depth

In [ ]:
# ===============================================================
# GENERATE ON VALIDATION SET
# ===============================================================

def generate_plan(model, tokenizer, row: dict, max_new_tokens: int = 2048) -> str:
    test_input = build_architect_input(row.get('input_payload', {}))
    messages = [
        {"role": "system", "content": ARCHITECT_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(test_input, indent=1, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CONFIG['max_seq_length'] - max_new_tokens)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.35,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )
    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def extract_json(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
    try:
        return json.loads(text)
    except:
        pass
    start = text.find('{')
    if start == -1:
        return None
    depth, in_string, escape_next = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except:
                    return None
    return None


# Get validation rows
val_sample_ids_set = set(sample_ids[i] for i in val_idx)
val_rows = [r for r in raw_data if r.get('sample_id', '') in val_sample_ids_set]

EVAL_SIZE = min(50, len(val_rows))
eval_sample = random.Random(SEED).sample(val_rows, EVAL_SIZE)

print(f"Generating on {EVAL_SIZE} validation examples...")

predictions = []
start_global = time.time()
f = open("training_output/validation_outputs.jsonl", "a", encoding="utf-8")

for i, row in enumerate(eval_sample):
    sid = row.get('sample_id', f'val_{i}')
    start_sample = time.time()

    try:
        gen_text = generate_plan(model, tokenizer, row)
        parsed = extract_json(gen_text)
    except Exception as e:
        print(f"  ERROR {sid}: {str(e)[:80]}")
        gen_text = ""
        parsed = None

    predictions.append({
        "sample_id": sid,
        "raw": gen_text[:800],
        "parsed": parsed,
        "reference": row.get('target_output', {}),
        "json_valid": parsed is not None,
        "case_type": row.get('metadata', {}).get('case_type', ''),
    })
    f.write(json.dumps(predictions[-1], ensure_ascii=False) + "\n")

    elapsed_sample = time.time() - start_sample
    elapsed_total = time.time() - start_global
    avg_time = elapsed_total / (i + 1)
    remaining = avg_time * (EVAL_SIZE - (i + 1))

    print(f"[{i+1}/{EVAL_SIZE}] {elapsed_sample:.1f}s | ETA: {remaining/60:.1f}m")

    if i < 3:
        print(f"\n  --- {sid} ({predictions[-1]['case_type']}) ---")
        print(f"  JSON valid: {parsed is not None}")
        if parsed:
            present = [k for k in MANDATORY_PLAN_KEYS if k in parsed]
            print(f"  Sections: {len(present)}/{len(MANDATORY_PLAN_KEYS)}")
            richness = compute_plan_richness(parsed)
            print(f"  Words: {richness['total_words']}, Tech terms: {richness['tech_term_count']}")
        else:
            print(f"  Raw: {gen_text[:200]}")
        print()

f.close()
json_valid_count = sum(1 for p in predictions if p['json_valid'])
print(f"\nDone. JSON valid: {json_valid_count}/{EVAL_SIZE} ({json_valid_count/EVAL_SIZE*100:.1f}%)")

In [ ]:
# ===============================================================
# COMPUTE ALL METRICS
# ===============================================================

n = len(predictions)

# ── Structural metrics ──────────────────────────────────────────────────────

json_valid_count = sum(1 for p in predictions if p['parsed'] is not None)

schema_ok = 0
fixrpt_ok = 0
fixrpt_total = 0
casetype_ok = 0
title_ok = 0

for pred in predictions:
    parsed    = pred['parsed']
    case_type = pred['case_type']
    if not parsed:
        continue

    # Schema: all mandatory sections present
    if all(k in parsed for k in MANDATORY_PLAN_KEYS):
        schema_ok += 1

    # Fix_report integrity (revision rounds only)
    if case_type == 'revision_round':
        fixrpt_total += 1
        fix_report = parsed.get('fix_report', [])
        # Get issue_ledger from the original row
        orig_row  = next((r for r in raw_data if r.get('sample_id') == pred['sample_id']), None)
        if orig_row:
            ledger_ids = set(orig_row.get('input_payload', {}).get('issue_ledger', {}).keys())
            if isinstance(fix_report, list) and fix_report:
                fr_ids = {item.get('issue_id','') for item in fix_report if isinstance(item, dict)}
                orphans = fr_ids - ledger_ids - {''}
                if not orphans:
                    fixrpt_ok += 1
            else:
                pass

    # Case type compliance
    fix_report_generated = parsed.get('fix_report', [])
    if case_type == 'first_pass' and (not fix_report_generated or fix_report_generated == []):
        casetype_ok += 1
    elif case_type == 'revision_round' and isinstance(fix_report_generated, list) and fix_report_generated:
        casetype_ok += 1

    # Title quality: no "Round N" patterns
    title = str(parsed.get('title', ''))
    if not re.search(r'\bround\s*\d', title, re.IGNORECASE):
        title_ok += 1

# ── Semantic metrics ─────────────────────────────────────────────────────────

plan_word_diffs = []
tech_term_counts = []
ref_word_counts = []
gen_word_counts = []
section_complete = []
section_jaccard = []
fixrpt_depth_scores = []

for pred in predictions:
    parsed = pred['parsed']
    ref = pred['reference']
    if not parsed or not ref:
        continue

    richness_gen = compute_plan_richness(parsed)
    richness_ref = compute_plan_richness(ref)

    gen_word_counts.append(richness_gen['total_words'])
    ref_word_counts.append(richness_ref['total_words'])
    plan_word_diffs.append(richness_gen['total_words'] - richness_ref['total_words'])
    tech_term_counts.append(richness_gen['tech_term_count'])

    # Section completeness
    complete = sum(
        1 for s in PLAN_CONTENT_SECTIONS
        if len(str(parsed.get(s, '')).split()) > 20
    )
    section_complete.append(complete / len(PLAN_CONTENT_SECTIONS))

    # Section-level Jaccard overlap
    jaccard_per_section = []
    for section in PLAN_CONTENT_SECTIONS:
        gen_text = str(parsed.get(section, '')).lower()
        ref_text = str(ref.get(section, '')).lower()
        gen_words_set = set(gen_text.split())
        ref_words_set = set(ref_text.split())
        intersection = gen_words_set & ref_words_set
        union = gen_words_set | ref_words_set
        jaccard_per_section.append(len(intersection) / max(len(union), 1))
    section_jaccard.append(np.mean(jaccard_per_section))

    # Fix_report depth
    for item in parsed.get('fix_report', []):
        if isinstance(item, dict):
            at_words = len(str(item.get('action_taken', '')).split())
            eo_words = len(str(item.get('expected_outcome', '')).split())
            fixrpt_depth_scores.append(at_words + eo_words)

# ── Report ────────────────────────────────────────────────────────────────────

structural = {
    "json_validity_rate": json_valid_count / n,
    "schema_compliance_rate": schema_ok / n,
    "fixreport_integrity_rate": fixrpt_ok / max(fixrpt_total, 1),
    "casetype_compliance_rate": casetype_ok / n,
    "title_quality_rate": title_ok / n,
}

print(f"\n{'='*60}")
print(f"STRUCTURAL METRICS (n={n})")
print(f"{'='*60}")
for k, v in structural.items():
    bar = '#' * int(v * 30)
    print(f"  {k:35s}: {v:6.1%}  {bar}")

print(f"\n{'='*60}")
print(f"SEMANTIC METRICS")
print(f"{'='*60}")
if gen_word_counts:
    print(f"  Generated plan words : mean={np.mean(gen_word_counts):.0f}  "f"median={np.median(gen_word_counts):.0f}")
    print(f"  Reference plan words : mean={np.mean(ref_word_counts):.0f}  "f"median={np.median(ref_word_counts):.0f}")
    print(f"  Word count diff : mean={np.mean(plan_word_diffs):+.0f}  "f"(generated vs reference)")
if tech_term_counts:
    print(f"  Tech term count : mean={np.mean(tech_term_counts):.1f}  "f"(reference: {np.mean([compute_plan_richness(p["reference"])["tech_term_count"] for p in predictions if p["reference"]]):.1f})")
if section_complete:
    print(f"  Section completeness : mean={np.mean(section_complete):.1%}  "f"(sections with >20 words)")
if section_jaccard:
    print(f"  Section Jaccard : mean={np.mean(section_jaccard):.3f}  "f"(word-overlap vs reference)")
if fixrpt_depth_scores:
    print(f"  Fix_report depth : mean={np.mean(fixrpt_depth_scores):.0f} words per item")

all_metrics = {
    "training": {
        "train_loss": train_result.training_loss,
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "duration_minutes": elapsed / 60,
        "trainable_params": trainable,
        "trainable_pct": pct,
    },
    "structural": structural,
    "semantic": {
        "mean_gen_words": float(np.mean(gen_word_counts)) if gen_word_counts else None,
        "mean_ref_words": float(np.mean(ref_word_counts)) if ref_word_counts else None,
        "mean_word_diff": float(np.mean(plan_word_diffs)) if plan_word_diffs else None,
        "mean_tech_terms": float(np.mean(tech_term_counts)) if tech_term_counts else None,
        "mean_section_complete": float(np.mean(section_complete)) if section_complete else None,
        "mean_section_jaccard": float(np.mean(section_jaccard)) if section_jaccard else None,
        "mean_fixreport_depth": float(np.mean(fixrpt_depth_scores)) if fixrpt_depth_scores else None,
    },
}

with open(OUTPUT_DIR / 'all_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"\nAll metrics saved to {OUTPUT_DIR / 'all_metrics.json'}")

---
## Section 12 — Results Visualisation

In [ ]:
# ===============================================================
# TRAINING CURVES
# ===============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if metrics_cb.train_losses:
    axes[0].plot(metrics_cb.steps, metrics_cb.train_losses, alpha=0.4, linewidth=0.8, color='#4C72B0', label='Raw')
    if len(metrics_cb.train_losses) > 10:
        w = min(20, len(metrics_cb.train_losses) // 3)
        smoothed = pd.Series(metrics_cb.train_losses).rolling(w, center=True).mean()
        axes[0].plot(metrics_cb.steps, smoothed, 'r-', linewidth=2, label=f'Smoothed (w={w})')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss (Completion-Only)')
    axes[0].legend()

if metrics_cb.eval_losses:
    axes[1].plot(metrics_cb.eval_steps_list, metrics_cb.eval_losses, 'go-', linewidth=2, markersize=6)
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss')

if metrics_cb.learning_rates:
    lr_steps = metrics_cb.steps[:len(metrics_cb.learning_rates)]
    axes[2].plot(lr_steps, metrics_cb.learning_rates, color='orange', linewidth=2)
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Cosine LR Schedule')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")

In [ ]:
# ===============================================================
# EVALUATION RESULTS
# ===============================================================

fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# 1. Structural metrics bar chart
ax1 = fig.add_subplot(gs[0, 0])
names = ['JSON\nValid', 'Schema', 'Fix_Rpt\nIntegrity', 'Case Type\nCompliance', 'Title\nQuality']
vals = list(structural.values())
colors = ['#55A868' if v > 0.85 else '#DD8452' if v > 0.60 else '#C44E52' for v in vals]
bars = ax1.bar(names, vals, color=colors, edgecolor='black', alpha=0.8)
ax1.set_ylim(0, 1.08)
ax1.set_ylabel('Compliance Rate')
ax1.set_title('Structural Output Quality')
for bar, val in zip(bars, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.0%}', ha='center', fontsize=9)

# 2. Generated vs Reference word count scatter
ax2 = fig.add_subplot(gs[0, 1])
if gen_word_counts and ref_word_counts:
    ax2.scatter(ref_word_counts, gen_word_counts, alpha=0.5, s=25, c='#8172B3')
    max_val = max(max(ref_word_counts), max(gen_word_counts))
    ax2.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='Perfect')
    ax2.set_xlabel('Reference Plan Words')
    ax2.set_ylabel('Generated Plan Words')
    ax2.set_title('Plan Richness: Generated vs Reference')
    if len(gen_word_counts) > 2:
        r_val, _ = stats.pearsonr(ref_word_counts, gen_word_counts)
        ax2.annotate(f'r={r_val:.3f}', xy=(0.05, 0.92), xycoords='axes fraction')
    ax2.legend()

# 3. Section-level Jaccard distribution
ax3 = fig.add_subplot(gs[0, 2])
if section_jaccard:
    ax3.hist(section_jaccard, bins=15, edgecolor='black', alpha=0.7, color='#4C72B0')
    ax3.axvline(np.mean(section_jaccard), color='red', linestyle='--', label=f'Mean={np.mean(section_jaccard):.3f}')
    ax3.set_xlabel('Section Jaccard (word overlap)')
    ax3.set_ylabel('Count')
    ax3.set_title('Section-Level Content Overlap')
    ax3.legend()

# 4. Section completeness distribution
ax4 = fig.add_subplot(gs[1, 0])
if section_complete:
    ax4.hist([s * 100 for s in section_complete], bins=10, edgecolor='black', alpha=0.7, color='#55A868')
    ax4.axvline(np.mean(section_complete) * 100, color='red', linestyle='--', label=f'Mean={np.mean(section_complete):.0%}')
    ax4.set_xlabel('% Sections with >20 Words')
    ax4.set_ylabel('Count')
    ax4.set_title('Section Completeness Rate')
    ax4.legend()

# 5. Tech term count: generated vs reference
ax5 = fig.add_subplot(gs[1, 1])
ref_tech = [compute_plan_richness(p['reference'])['tech_term_count'] for p in predictions if p['reference']]
gen_tech = tech_term_counts
if ref_tech and gen_tech:
    ax5.scatter(ref_tech, gen_tech[:len(ref_tech)], alpha=0.5, s=25, c='#DD8452')
    max_t = max(max(ref_tech), max(gen_tech))
    ax5.plot([0, max_t], [0, max_t], 'r--', alpha=0.5, label='Perfect')
    ax5.set_xlabel('Reference Tech Term Count')
    ax5.set_ylabel('Generated Tech Term Count')
    ax5.set_title('Technology Specificity')
    ax5.legend()

# 6. Summary metrics table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
summary_rows = [
    ['Training Loss', f'{train_result.training_loss:.4f}'],
    ['Eval Loss', f'{eval_loss:.4f}'],
    ['Perplexity', f'{perplexity:.2f}'],
    ['Training Time', f'{elapsed/60:.1f} min'],
    ['Trainable Params', f'{trainable:,} ({pct:.2f}%)'],
    ['JSON Validity', f'{structural["json_validity_rate"]:.1%}'],
    ['Schema Compliance', f'{structural["schema_compliance_rate"]:.1%}'],
    ['Fix_Rpt Integrity', f'{structural["fixreport_integrity_rate"]:.1%}'],
    ['Mean Gen Words', f'{np.mean(gen_word_counts):.0f}' if gen_word_counts else 'N/A'],
    ['Section Jaccard', f'{np.mean(section_jaccard):.3f}' if section_jaccard else 'N/A'],
    ['Section Complete', f'{np.mean(section_complete):.1%}' if section_complete else 'N/A'],
    ['Mean Tech Terms', f'{np.mean(tech_term_counts):.1f}' if tech_term_counts else 'N/A'],
]
table = ax6.table(cellText=summary_rows, colLabels=['Metric', 'Value'], loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.7)
ax6.set_title('Comprehensive Results Summary', fontsize=12, fontweight='bold', pad=20)

plt.savefig(OUTPUT_DIR / 'evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: evaluation_results.png")

---
## Section 13 — Save Merged Model & Training Record

In [ ]:
# ===============================================================
# SAVE MERGED MODEL
# ===============================================================

SAVE_MERGED = True

if SAVE_MERGED:
    print("Merging adapter into base model...")
    merged_dir = FINAL_MODEL_DIR / "merged_model"
    merged_dir.mkdir(exist_ok=True)

    merged_model = model.merge_and_unload()

    for name, module in merged_model.named_modules():
        tied = getattr(module, "_tied_weights_keys", None)
        if isinstance(tied, list):
            module._tied_weights_keys = {k: k for k in tied}

    merged_model.save_pretrained(str(merged_dir), safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(str(merged_dir))

    merged_size = sum(f.stat().st_size for f in merged_dir.rglob('*') if f.is_file()) / 1e9
    print(f"Merged model saved: {merged_dir} ({merged_size:.2f} GB)")
else:
    print("Skipped merged model (SAVE_MERGED=False)")

In [ ]:
# ===============================================================
# SAVE TRAINING RECORD
# ===============================================================

record = {
    "model_name": MODEL_NAME,
    "dataset_path": DATASET_PATH,
    "dataset_size": len(raw_data),
    "formatted_size": len(formatted_texts),
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "config": CONFIG,
    "trainable_parameters": trainable,
    "trainable_percentage": pct,
    "training_loss": train_result.training_loss,
    "eval_loss": eval_loss,
    "perplexity": perplexity,
    "training_duration_minutes": elapsed / 60,
    "structural_metrics": structural,
    "semantic_metrics": all_metrics.get("semantic", {}),
    "completion_only_masking": True,
    "plan_quality_fix_applied": True,
    "plan_quality_fix_description": (
        "plan_quality metadata excluded from training format entirely. "
        "Richness filter applied at load time (total_words >= 400, zero stub sections). "
        "This replaces the unreliable strong/weak/flawed -> good/moderate proxy labels."
    ),
    "seed": SEED,
}

with open(FINAL_MODEL_DIR / 'training_record.json', 'w') as f:
    json.dump(record, f, indent=2, default=str)

print(f"Training record saved to {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"\n{'='*60}")
print("ALL OUTPUTS:")
print(f"  Adapter : {FINAL_MODEL_DIR / 'lora_adapter'}")
if SAVE_MERGED:
    print(f"  Merged model : {FINAL_MODEL_DIR / 'merged_model'}")
print(f"  Curves : {OUTPUT_DIR / 'training_curves.png'}")
print(f"  Evaluation : {OUTPUT_DIR / 'evaluation_results.png'}")
print(f"  Metrics : {OUTPUT_DIR / 'all_metrics.json'}")
print(f"  Record : {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"  TensorBoard : {LOGS_DIR}")
print(f"{'='*60}")

In [ ]:
import shutil
shutil.make_archive('/workspace/training_output', 'zip', '/workspace/training_output')

---
## Summary

This notebook fine-tuned Phi-3-mini-128k-instruct using QLoRA with **completion-only loss masking**
to serve as the ArchitectAgent in an architectural planning multi-agent system.

### Key Technical Contributions

1. **Completion-only loss masking** via `DataCollatorForCompletionOnlyLM` ensures the model
   learns to produce implementation-grade 19-section architecture plans, not predict the
   deterministic input contract

2. **Structured input trimming** (`trim_contract`, `trim_specialist_subplans`, `trim_plan_reference`)
   reduces the median input from ~7700 to ~4800 tokens while preserving all semantically
   critical fields; `frozen_requirement_contract` is trimmed to values-only since source,
   rationale and timestamps are pipeline metadata not needed at inference time

3. **plan_quality fix** — the unreliable `strong`/`weak`/`flawed` labels (and their
   word-count-derived replacements) are excluded entirely from the training format.
   A richness filter (`total_words >= 400`, zero stub sections) applied at load time
   provides a genuine quality gate using the actual plan content

4. **Chain-aware data splitting** prevents leakage; each row is self-contained with its
   full revision context embedded in `input_payload`

5. **Architect-specific evaluation**: section completeness rate, section-level Jaccard
   overlap, fix_report integrity (issue IDs reference the input ledger), and tech term
   specificity — complementing the standard JSON validity and schema compliance checks

### How to Load the Model for Inference

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Option A: LoRA adapter (small, needs base model)
base  = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-128k-instruct")
model = PeftModel.from_pretrained(base, "training_output/architect_agent_model/lora_adapter")

# Option B: Merged model (standalone)
model = AutoModelForCausalLM.from_pretrained("training_output/architect_agent_model/merged_model")
```